In [1]:
!pip install -U langchain-openrouter langchain langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.0 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.17
    Uninstalling langchain-1.3.17:
      Successfully uninstalled langchain-1.3.17


In [2]:
import os
from typing import Annotated, TypedDict

from google.colab import userdata

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
    AIMessage
)

from langchain_core.tools import tool

from langchain_openrouter import ChatOpenRouter

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


# ---------------------------------------------------------------------------
# 0. API Key & Nemotron Model Setup
# ---------------------------------------------------------------------------

try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except Exception as e:
    raise RuntimeError(
        "Please add 'OPENROUTER_API_KEY' to your Colab Secrets "
        "(Key icon on the left sidebar) and enable notebook access."
    ) from e


# ---------------------------------------------------------------------------
# Nemotron Model
# ---------------------------------------------------------------------------

llm = ChatOpenRouter(
    model="nvidia/nemotron-3.5-lightning:free",
    temperature=0
)


# ---------------------------------------------------------------------------
# Helper function
# ---------------------------------------------------------------------------

def extract_text_safely(content) -> str:

    if isinstance(content, str):
        return content

    elif isinstance(content, list):

        parts = []

        for item in content:

            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])

            elif isinstance(item, str):
                parts.append(item)

            else:
                parts.append(str(item))

        return "".join(parts)

    return str(content)


# ---------------------------------------------------------------------------
# 1. State Definition
# ---------------------------------------------------------------------------

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_node: str


# ---------------------------------------------------------------------------
# 2. Tool Definition
# ---------------------------------------------------------------------------

@tool
def process_refund(user_id: str, amount: float) -> str:
    """Executes a financial refund for a specific user ID."""

    return (
        f"SUCCESS: Refund of ${amount} has been processed "
        f"for User '{user_id}'."
    )


# ---------------------------------------------------------------------------
# 3. Agent Nodes
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Node A: Supervisor Router Agent
# ---------------------------------------------------------------------------

def supervisor_agent(state: AgentState) -> AgentState:

    system_prompt = (
        "You are a Support Router. Analyze the user prompt:\n"
        "- If it is general technical troubleshooting, respond with 'SUPPORT'.\n"
        "- If it involves financial refunds or account modifications, "
        "respond with 'ACTION'.\n"
        "Respond ONLY with 'SUPPORT' or 'ACTION'."
    )

    messages = [
        SystemMessage(content=system_prompt)
    ] + state["messages"]

    response = llm.invoke(messages)

    raw_text = extract_text_safely(response.content)

    decision = raw_text.strip().upper()

    if "ACTION" in decision:

        next_step = "account_actions_agent"

    elif "SUPPORT" in decision:

        next_step = "tech_support_agent"

    else:

        next_step = END

    return {
        "next_node": next_step
    }


# ---------------------------------------------------------------------------
# Node B: Technical Support Agent
# ---------------------------------------------------------------------------

def tech_support_agent(state: AgentState) -> AgentState:

    system_prompt = SystemMessage(
        content=(
            "You are a helpful Technical Support Specialist. "
            "Provide concise troubleshooting guidance."
        )
    )

    messages = [
        system_prompt
    ] + state["messages"]

    response = llm.invoke(messages)

    text_content = extract_text_safely(response.content)

    return {
        "messages": [
            AIMessage(
                content=f"[Tech Support]: {text_content}"
            )
        ],
        "next_node": END
    }


# ---------------------------------------------------------------------------
# Node C: Account Action Agent
# ---------------------------------------------------------------------------

def account_actions_agent(state: AgentState) -> AgentState:

    llm_with_tools = llm.bind_tools(
        [process_refund]
    )

    system_prompt = SystemMessage(
        content=(
            "You are an Account Manager. "
            "Use the process_refund tool to issue user refunds "
            "when requested."
        )
    )

    messages = [
        system_prompt
    ] + state["messages"]

    response = llm_with_tools.invoke(messages)


    # Check if Nemotron requested a tool call
    if response.tool_calls:

        tool_call = response.tool_calls[0]

        tool_output = process_refund.invoke(
            tool_call["args"]
        )

        final_msg = (
            f"[Account Agent]: Executed tool. "
            f"Result: {tool_output}"
        )

    else:

        text_content = extract_text_safely(
            response.content
        )

        final_msg = (
            f"[Account Agent]: {text_content}"
        )


    return {
        "messages": [
            AIMessage(content=final_msg)
        ],
        "next_node": END
    }


# ---------------------------------------------------------------------------
# 4. Construct LangGraph Workflow
# ---------------------------------------------------------------------------

workflow = StateGraph(AgentState)


# Add nodes
workflow.add_node(
    "supervisor",
    supervisor_agent
)

workflow.add_node(
    "tech_support_agent",
    tech_support_agent
)

workflow.add_node(
    "account_actions_agent",
    account_actions_agent
)


# START → Supervisor
workflow.add_edge(
    START,
    "supervisor"
)


# Supervisor → appropriate agent
workflow.add_conditional_edges(

    "supervisor",

    lambda state: state["next_node"],

    {
        "tech_support_agent": "tech_support_agent",
        "account_actions_agent": "account_actions_agent",
        END: END
    }
)


# Agents → END
workflow.add_edge(
    "tech_support_agent",
    END
)

workflow.add_edge(
    "account_actions_agent",
    END
)


# Compile workflow
app = workflow.compile()


# ---------------------------------------------------------------------------
# 5. Live Demonstration Function
# ---------------------------------------------------------------------------

def run_demo(user_query: str):

    print(
        f"\n================ USER QUERY ================\n"
        f"{user_query}"
    )

    inputs = {
        "messages": [
            HumanMessage(content=user_query)
        ]
    }

    result = app.invoke(inputs)

    print(
        "\n================ SYSTEM RESPONSE ================"
    )

    print(
        result["messages"][-1].content
    )


# ---------------------------------------------------------------------------
# 6. Test Cases
# ---------------------------------------------------------------------------


# Test Case 1: Technical Support
run_demo(
    "My app keeps freezing whenever I try to upload a PNG file. "
    "How can I fix this?"
)


# Test Case 2: Financial Refund
run_demo(
    "I was billed twice by mistake. "
    "Please refund $49.99 for my account 'user_9876'."
)


================ USER QUERY ================
My app keeps freezing whenever I try to upload a PNG file. How can I fix this?

================ SYSTEM RESPONSE ================
[Tech Support]: **Quick troubleshooting steps:**

1. **Check file size** – Very large PNGs (e.g., >10–20 MB) can overwhelm memory. Try a smaller file.
2. **Restart the app/device** – Clears temporary glitches.
3. **Update the app** – Ensure you’re on the latest version; PNG-handling bugs are often fixed in updates.
4. **Clear cache** – If it’s a web app, clear your browser cache/cookies.
5. **Test with a different PNG** – A corrupted file may cause the freeze. Try a known-good PNG or convert to JPEG.
6. **Disable browser extensions/add-ons** – Some ad/tracking blockers interfere with file uploads.
7. **Check OS/app compatibility** – Ensure your OS version supports the app’s PNG features.

**If it persists, reply with:**  
- App name & version  
- Device/OS  
- PNG file size & whether it has transparency/large dim